# Chapter 7 Simulations — Grid Search, Ground-Truth Evaluation & Judge Calibration

This notebook runs three evaluations against the three-agent collection pipeline defined in `my_agent/agent.py`:

1. **Grid Search** — three models × three temperatures × ten QIRs (from `Additional QIRs.xlsx`) × five Monte Carlo simulations = 450 runs, executed as direct model calls. Ranked by F1 (judge-compliance metric).
2. **Ground-Truth Evaluation** — three expert-curated QIRs from `ground_truth.csv`. Pipeline output is compared against the analyst's source picks via Essential Coverage.
3. **Judge Calibration** — ten synthetic plans with known-correct verdicts test whether the judge correctly classifies hallucinated sources, near-miss names, subtle product disambiguation, and tangential sources.

**Why all three?** Grid search measures how well the agent satisfies the judge. Ground-truth measures whether the plan matches what a human analyst would write. Judge calibration measures whether the judge itself can be trusted. A lenient judge produces F1=100% on mediocre output — only the ground-truth and calibration tests catch that.

## 1. Setup

Load environment variables and import the pipeline. The agents are constructed once at import time; the notebook re-uses them across all runs.


In [ ]:
import asyncio
import json
import os
import sys
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from dotenv import load_dotenv

# Load API key — walks up the directory tree to find .env (works from any
# subdirectory). Falls back to Colab Secrets if not found.
load_dotenv()
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass  # Not in Colab or secret not configured

# Make my_agent importable
sys.path.insert(0, str(Path("my_agent").resolve()))

from agent import (
    build_pipeline,
    JudgeVerdict,
    APP_NAME,
    USER_ID,
)
from sources import KNOWN_SOURCES
from google.adk import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Build the full set of catalog source names once at import time.
# Used by extract_sources_from_plan() to decouple ground-truth evaluation
# from the judge verdict entirely.
ALL_CATALOG_SOURCES = {
    s["name"]
    for section in KNOWN_SOURCES.values()
    for s in section
}


## 2. Pipeline Helper

Wraps a single end-to-end run of the LoopAgent. Each call uses `build_pipeline()` from `agent.py`, which constructs fresh agent instances. ADK sets a parent reference on every sub-agent when it is added to a parent — reusing the same instances across calls would raise "Agent already has a parent".

In [ ]:
import gc
import time

async def run_pipeline(qir: str, threat_context: str, max_iterations: int = 3,
                       collection_temp: float = 0.0,
                       collection_model: str = "gemini-2.5-flash") -> dict:
    """Run the three-agent pipeline once and return final state + metrics.

    `collection_model` / `collection_temp` set the Collection (generator) agent;
    the grid search sweeps these. The judge and verification agents keep their
    fixed models. Latency is measured for the Collection agent's FIRST generation
    only (one generation), since the grid tunes that agent's parameters, not the
    whole pipeline.
    """
    # Time only the Collection agent's first generation, via before/after callbacks.
    _timer = {"start": None, "latency_s": None}
    def _coll_before(callback_context):
        if _timer["start"] is None:
            _timer["start"] = time.monotonic()
        return None
    def _coll_after(callback_context):
        if _timer["latency_s"] is None and _timer["start"] is not None:
            _timer["latency_s"] = round(time.monotonic() - _timer["start"], 1)
        return None

    loop = build_pipeline(collection_temp=collection_temp,
                          collection_model=collection_model,
                          max_iterations=max_iterations,
                          collection_before_callback=_coll_before,
                          collection_after_callback=_coll_after)

    session_service = InMemorySessionService()
    session_id = f"sim_{uuid.uuid4().hex}"
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id,
        state={"threat_context": threat_context, "verified_plan": "",
               "validation_verdict": ""},
    )

    runner = Runner(agent=loop, app_name=APP_NAME, session_service=session_service)
    iters_seen = set()
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=qir)]),
    ):
        if event.author == "Collection_Recommendation_Agent":
            iters_seen.add(id(event))

    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id,
    )

    verdict_raw = session.state.get("validation_verdict", "{}")
    try:
        verdict = JudgeVerdict.model_validate(json.loads(verdict_raw)).model_dump()
    except Exception:
        verdict = {"confirmed_valid": [], "unverified": [],
                   "missing_critical": [], "verdict": "FAIL",
                   "summary": "Verdict parse failed."}

    result = {
        "verdict":               verdict,
        "verified_plan":         session.state.get("verified_plan", ""),
        "iters":                 max(1, len(iters_seen)),
        "collection_latency_s":  _timer["latency_s"],
    }

    # Explicitly release large objects before returning.
    # Each pipeline holds 3 agents with the full 56-source catalog in their
    # instructions — without cleanup these accumulate and crash Colab.
    del runner, loop, session_service, session
    gc.collect()

    return result

## 3. Compute Metrics

Four metrics are derived from the final iteration's verdict:

- **Precision** = `valid / (valid + unverified)` × 100 — catalog hit rate
- **Coverage** = `valid / (valid + missing)` × 100 — fraction of critical sources captured
- **F1** = harmonic mean of the two
- **Iterations** — how many loops until convergence

These are *judge-compliance* metrics, not absolute correctness. The ground-truth section below provides the external reference.


In [ ]:
def compute_metrics(result: dict) -> dict:
    v = result["verdict"]
    n_v = len(v.get("confirmed_valid", []))
    n_u = len(v.get("unverified", []))
    n_m = len(v.get("missing_critical", []))

    precision = (n_v / (n_v + n_u) * 100) if (n_v + n_u) else 100.0
    coverage  = (n_v / (n_v + n_m) * 100) if (n_v + n_m) else 100.0
    f1 = (2 * precision * coverage / (precision + coverage)
          if (precision + coverage) else 0.0)

    return {
        "precision": round(precision, 1),
        "coverage":  round(coverage,  1),
        "f1":        round(f1,        1),
        "iters":     result["iters"],
        "passed":    v.get("verdict") == "PASS",
    }


## 4. Grid Search

The grid search and Monte Carlo techniques were introduced in Chapters 4 and 5 and first applied in Chapter 6's evaluation. The same infrastructure is available here — run the cell below to evaluate model and temperature configurations against your own API tier and QIR set.

The sweep covers three models (gemini-2.5-flash, gemini-2.5-pro, gemini-3.1-pro-preview) × three temperatures (0.0, 0.5, 1.0) × the ten QIRs in `Additional QIRs.xlsx` × five Monte Carlo simulations: **450 runs across nine configurations**. The grid calls the models **directly** with the same Collection, Judge, and Verification prompts the ADK agents use — it reproduces the refinement loop without the agent framework, so it isolates model behaviour. Only the Collection (generator) agent's model and temperature change; the judge stays fixed at gemini-3.1-pro-preview so it is a constant evaluator. **Latency is recorded for the Collection agent's single (first) generation only**, since the grid is tuning that agent's parameters; the judge and verification time is deliberately excluded. F1, Precision, and Coverage come from the judge verdict at convergence. All configurations are expected to return F1 near 100%: the collection task is constrained enough by the 56-source catalog and the judge's validation rules that model capability is not the limiting factor. Where all configurations saturate the metric, cost and latency favour `gemini-2.5-flash` at temperature 0.0.

The 450 runs are launched as concurrent `asyncio` coroutines and consumed with `asyncio.as_completed()` as they finish. A shared `asyncio.Semaphore(GRID_CONCURRENCY)` caps how many runs are active at once. The three calls *within* a run stay sequential (each pass depends on the previous verdict), so each active run has at most one call in flight and `GRID_CONCURRENCY = 20` caps the sweep at **20 peak concurrent API calls** — raising the cap raises concurrency one for one; independent runs overlap.

> ⏱️ **Runtime / rate-limit note.** Even concurrent, the full 450-run sweep is a sizeable job (the `gemini-2.5-pro` / `gemini-3.1-pro-preview` rows dominate) — budget **roughly 1–2 hours** depending on your API tier. Set `GRID_CONCURRENCY` so your requests per minute stay inside your API tier and lower it if you see 429 rate-limit errors. Validate on a subset first (slice `MODELS`, `TEMPERATURES`, or `QIRS`) before launching all 450.

In [ ]:
import re

# The grid sweeps the Collection (generator) agent across models, temperatures,
# and the ten QIRs in "Additional QIRs.xlsx"; the judge stays fixed
# (gemini-3.1-pro-preview). Each QIR is column A of that file; its threat context
# is built from the file's rationale columns (decision driver, available
# telemetry, relevance, timeframe). Ten QIRs x three models x three temperatures
# x five Monte Carlo simulations = 450 runs across nine configurations.
QIR_FILE = "Additional QIRs.xlsx"

def _clean(x):
    return str(x).replace("\xa0", " ").strip() if x is not None else ""

def load_qirs(path=QIR_FILE):
    """Return [(qir_text, threat_context), ...] from the additional-QIRs workbook."""
    raw = pd.read_excel(path, header=0)
    qirs = []
    for _, r in raw.iterrows():
        if not _clean(r.iloc[0]):
            continue
        qir_text = re.sub(r"^\s*\d+\)\s*", "", _clean(r.iloc[0]))   # drop "1) " prefix
        threat = (f"Decision driver: {_clean(r.iloc[1])} "
                  f"Available telemetry: {_clean(r.iloc[2])} "
                  f"Relevance: {_clean(r.iloc[3])} "
                  f"Timeframe: {_clean(r.iloc[4])}").strip()
        qirs.append((qir_text, threat))
    return qirs

QIRS = load_qirs()

MODELS         = ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-pro-preview"]
TEMPERATURES   = [0.0, 0.5, 1.0]
N_SIMULATIONS  = 5

# Concurrency cap for the grid: at most this many runs are active at once. The
# calls within a run are strictly sequential (each awaits the previous), so each
# active run has at most one API call in flight and this value IS the peak
# concurrent-call count. Set it so your requests per minute stay inside your
# API tier and lower it if you see 429 rate-limit errors.
GRID_CONCURRENCY = 20

n_configs  = len(MODELS) * len(TEMPERATURES)
total_runs = n_configs * len(QIRS) * N_SIMULATIONS
print(f"Loaded {len(QIRS)} QIRs from {QIR_FILE!r}")
print(f"Grid: {len(MODELS)} models x {len(TEMPERATURES)} temperatures x "
      f"{len(QIRS)} QIRs x {N_SIMULATIONS} sims = {total_runs} runs "
      f"across {n_configs} configurations")

# Runtime note: runs execute concurrently, capped at GRID_CONCURRENCY active runs
# (which is also the peak API-call count). The full 450-run sweep is still a sizeable job
# (the pro-tier models dominate); budget roughly 1-2 hours depending on your API
# tier, and lower GRID_CONCURRENCY if 429s appear. Slice MODELS/TEMPERATURES/QIRS
# to validate on a subset before committing to the full sweep.
print(f"\n  Concurrency: up to {GRID_CONCURRENCY} active runs ({GRID_CONCURRENCY} "
      f"peak API calls). Full sweep ~1-2h; lower GRID_CONCURRENCY on 429 errors.")

In [ ]:
# NOTE: REQUIRES API KEY - grid search: 450 runs, ~1-2 hours. The canonical
# results ship in config_search_results_original.csv if you want to skip the spend.

# Direct-model-call grid (no ADK), run CONCURRENTLY. The grid reproduces the
# Collection -> Judge -> Verification refinement loop by calling the models
# directly with the SAME prompts the ADK agents use. The 450 runs are launched as
# asyncio coroutines and consumed with as_completed; a shared asyncio.Semaphore caps how many runs
# are active at once (the calls within a run are sequential, so at most one call
# per run is in flight: Semaphore(20) = 20 peak concurrent calls). The judge stays fixed
# at JUDGE_MODEL; only the Collection agent's model and temperature change. Latency
# is the Collection agent's FIRST generation only (the parameter the grid tunes).
import asyncio
from google import genai
from google.genai import types as genai_types
from agent import (COLLECTION_INSTRUCTION, JUDGE_INSTRUCTION,
                   VERIFICATION_INSTRUCTION, DEFAULT_MODEL, JUDGE_MODEL)

_client = genai.Client()  # reads GOOGLE_API_KEY from the environment

def _fill(template: str, **kw) -> str:
    """Substitute {name} placeholders without str.format (catalog text has braces)."""
    out = template
    for k, v in kw.items():
        out = out.replace("{" + k + "}", v)
    return out

async def _collection_call(qir, threat_context, verified_plan, validation_verdict,
                           model, temperature):
    prompt = (_fill(COLLECTION_INSTRUCTION, threat_context=threat_context,
                    verified_plan=verified_plan, validation_verdict=validation_verdict)
              + "\n\n## Analyst QIR\n" + qir)
    cfg = genai_types.GenerateContentConfig(temperature=temperature, top_p=0.9,
                                            top_k=20, frequency_penalty=0.0,
                                            presence_penalty=0.0)
    t0 = time.monotonic()
    resp = await _client.aio.models.generate_content(model=model, contents=prompt, config=cfg)
    return (resp.text or ""), round(time.monotonic() - t0, 1)

async def _judge_call(collection_plan, validation_verdict):
    prompt = _fill(JUDGE_INSTRUCTION, collection_plan=collection_plan,
                   validation_verdict=validation_verdict)
    cfg = genai_types.GenerateContentConfig(temperature=0.0,
                                            response_mime_type="application/json")
    resp = await _client.aio.models.generate_content(model=JUDGE_MODEL, contents=prompt, config=cfg)
    return resp.text or "{}"

async def _verification_call(collection_plan, validation_verdict):
    prompt = _fill(VERIFICATION_INSTRUCTION, collection_plan=collection_plan,
                   validation_verdict=validation_verdict)
    cfg = genai_types.GenerateContentConfig(temperature=0.0, top_p=0.95, top_k=40)
    resp = await _client.aio.models.generate_content(model=DEFAULT_MODEL, contents=prompt, config=cfg)
    return resp.text or ""

_FAIL_VERDICT = {"confirmed_valid": [], "unverified": [], "missing_critical": [],
                 "verdict": "FAIL", "summary": "Verdict parse failed."}

async def run_config_direct(qir, threat_context, model, temperature, max_iterations=3):
    """Collection -> Judge -> Verification refinement loop via direct model calls.

    Mirrors build_pipeline's LoopAgent (max 3 iterations, exit on PASS) without
    invoking ADK. The three calls within a run are sequential (each pass depends on
    the previous verdict); independent runs execute concurrently (see run_grid_search).
    """
    verified_plan, verdict_raw, coll_latency, iters = "", "", None, 0
    verdict = dict(_FAIL_VERDICT, summary="No run.")
    for _ in range(max_iterations):
        iters += 1
        plan, lat = await _collection_call(qir, threat_context, verified_plan,
                                           verdict_raw, model, temperature)
        if coll_latency is None:
            coll_latency = lat                 # FIRST generation only
        verdict_raw = await _judge_call(plan, verdict_raw)
        try:
            verdict = JudgeVerdict.model_validate_json(verdict_raw).model_dump()
        except Exception:
            verdict = dict(_FAIL_VERDICT)
            break
        verified_plan = await _verification_call(plan, verdict_raw)
        if verdict.get("verdict") == "PASS":   # loop-exit: escalate on PASS
            break
    return {"verdict": verdict, "iters": iters, "collection_latency_s": coll_latency}

async def run_grid_search(concurrency=GRID_CONCURRENCY):
    sem = asyncio.Semaphore(concurrency)

    async def _one(model, temp, qi, qir, threat, sim):
        async with sem:                        # cap active runs (~3 calls each)
            try:
                result = await run_config_direct(qir, threat, model, temp)
                m = compute_metrics(result)
                m["latency_s"] = result["collection_latency_s"]
            except Exception as exc:
                print(f"  {model} T={temp} QIR{qi} sim{sim} FAILED: {exc}")
                m = {"precision": 0, "coverage": 0, "f1": 0, "iters": 3,
                     "passed": False, "latency_s": 0}
            return {"Model": model, "Temperature": temp, "QIR": qi, "Sim": sim, **m}

    tasks = [_one(model, temp, qi, qir, threat, sim)
             for model in MODELS
             for temp in TEMPERATURES
             for qi, (qir, threat) in enumerate(QIRS, start=1)
             for sim in range(1, N_SIMULATIONS + 1)]

    rows = []
    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="runs"):
        rows.append(await fut)

    order = {m: i for i, m in enumerate(MODELS)}
    return (pd.DataFrame(rows)
              .sort_values(["Model", "Temperature", "QIR", "Sim"],
                           key=lambda s: s.map(order) if s.name == "Model" else s)
              .reset_index(drop=True))

df_grid = await run_grid_search()
df_grid

### Grid search results — aggregated by configuration

The cell below aggregates the raw runs into one row per configuration (model × temperature), ranks them by average F1 (ties broken by iterations then latency), and writes `config_search_results.csv`. Near-uniform F1 across all configurations indicates benchmark saturation — the task is not difficult enough to differentiate models or temperatures on quality alone.

In [ ]:
# Aggregate the raw runs into one row per configuration (model x temperature),
# rank them, and persist to config_search_results.csv so the styled table and
# Figure 7.5 are regenerated from a single source of truth.
agg = (df_grid.groupby(["Model", "Temperature"])
                .agg(**{
                    "avg F1 (%)":        ("f1", "mean"),
                    "F1 Std Dev":        ("f1", "std"),
                    "avg Precision (%)": ("precision", "mean"),
                    "avg Coverage (%)":  ("coverage", "mean"),
                    "avg Iters":         ("iters", "mean"),
                    "Pass Rate (%)":     ("passed", lambda s: s.mean() * 100),
                    "avg Latency (s)":   ("latency_s", "mean"),
                })
                .reset_index())

agg["F1 Std Dev"] = agg["F1 Std Dev"].fillna(0.0)

# Rank by avg F1 (desc); ties broken by avg iterations then latency (both asc).
agg = (agg.sort_values(["avg F1 (%)", "avg Iters", "avg Latency (s)"],
                       ascending=[False, True, True])
          .reset_index(drop=True))
agg.insert(0, "Rank", range(1, len(agg) + 1))

agg = agg.round({"avg F1 (%)": 1, "F1 Std Dev": 1, "avg Precision (%)": 1,
                 "avg Coverage (%)": 1, "avg Iters": 2, "Pass Rate (%)": 0,
                 "avg Latency (s)": 1})

agg.to_csv("config_search_results.csv", index=False)
agg

### Full config search results — all 9 configurations

Results from the 450-run grid search (3 models × 3 temperatures × 10 QIRs × 5 Monte Carlo simulations), executed as direct model calls. Ranked by avg F1, ties broken by avg iterations. ★ marks the selected configuration.

`best_config.json` does not ship with the repository — it stays empty until you run the code: the cell after the ranked table writes the ★ configuration to `best_config.json` in this folder, from the canonical ranked results loaded above (`df_configs`). If you re-ran the grid, your own ranking lives in `config_search_results.csv`.

In [ ]:
# Read the canonical 450-run results the chapter references
# (config_search_results_original.csv). The aggregation cell above writes a
# separate config_search_results.csv as a reproduction target, so re-running the
# grid never clobbers this canonical file.
df_configs = pd.read_csv("config_search_results_original.csv")

def _style_configs(df):
    def highlight_best(row):
        return ["background-color: #dcfce7; font-weight: bold" if row["★"] == "★"
                else "" for _ in row]

    def color_f1(val):
        if val == 100.0:
            return "color: #15803d; font-weight: 600"
        elif val >= 99.5:
            return "color: #ca8a04"
        else:
            return "color: #dc2626"

    def color_pass(val):
        if val == 100:
            return "color: #15803d; font-weight: 600"
        elif val >= 96:
            return "color: #ca8a04"
        else:
            return "color: #dc2626"

    display_df = df.copy()
    display_df.insert(0, "★", display_df["Rank"].apply(lambda r: "★" if r == 1 else ""))
    display_df = display_df.drop(columns=["Rank"])
    display_df = display_df.rename(columns={
        "avg F1 (%)":        "avg F1 %",
        "F1 Std Dev":        "± std",
        "avg Precision (%)": "Precision %",
        "avg Coverage (%)":  "Coverage %",
        "avg Iters":         "Iters",
        "Pass Rate (%)":     "Pass %",
        "avg Latency (s)":   "Latency (s)",
    })

    styler = display_df.style.apply(highlight_best, axis=1)
    # applymap (pandas <2.1) was renamed to map in pandas 2.1+
    _map = styler.map
    styler = _map(color_f1,   subset=["avg F1 %"])
    styler = _map(color_pass, subset=["Pass %"])
    return (
        styler
        .format({
            "avg F1 %":    "{:.1f}",
            "± std":       "{:.1f}",
            "Precision %": "{:.1f}",
            "Coverage %":  "{:.1f}",
            "Iters":       "{:.2f}",
            "Pass %":      "{:.0f}",
            "Latency (s)": "{:.1f}",
        })
        .set_caption(
            "Config search results — 3 models × 3 temperatures × 10 QIRs × 5 Monte Carlo simulations = 450 runs"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.85rem"), ("color", "#64748b"),
                       ("padding-bottom", "8px"), ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )

_style_configs(df_configs)

In [ ]:
# Write the selected (Rank-1) configuration to best_config.json.
# The repo does not ship this file — it stays empty until this cell runs, which
# creates it from the canonical ranked results loaded above (df_configs). If
# you re-ran the grid, your own ranking lives in config_search_results.csv.
best_row = df_configs.loc[df_configs["Rank"] == 1].iloc[0]
best_config = {"model": best_row["Model"], "temperature": float(best_row["Temperature"])}
with open("best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)
print(f"best_config.json written: {best_config}")

### Visualization — F1 distribution per temperature

Box plot with scatter overlay. The y-axis is inverted so the highest F1 (best) sits at the top.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_grid, x="Temperature", y="f1", hue="Model",
            linewidth=1.5, width=0.6, showfliers=False)
sns.stripplot(data=df_grid, x="Temperature", y="f1", hue="Model",
              dodge=True, alpha=0.5, jitter=0.1, size=6, legend=False)
plt.title(f"Grid Search: F1 distribution across {N_SIMULATIONS} simulations "
          f"per model x temperature", fontsize=14)
plt.ylabel("F1 (higher is better)", fontsize=12)
plt.xlabel("Collection Agent Temperature", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Ground-Truth Evaluation

Loads three expert-curated QIRs from `ground_truth.csv`, runs the full pipeline against each, and compares the agent's output against the analyst's expected source selections. Sources are extracted directly from the `verified_plan` text — decoupled from the judge verdict entirely — so the score is independent of whether the judge is well-calibrated.

The primary metric is **Essential Coverage**: the fraction of expert-designated essential sources that appear in the final plan. A score below 100% means the pipeline missed at least one source a senior analyst would consider critical for that threat.

In [ ]:
gt_df = pd.read_csv("ground_truth.csv")

In [ ]:
def extract_sources_from_plan(plan_text: str) -> set[str]:
    """Return catalog source names that appear in the verified plan text.

    This is intentionally decoupled from the judge verdict. The judge's
    confirmed_valid list reflects the judge's opinion; this function reads
    the plan text directly, so ground-truth accuracy is independent of
    whether the judge is well-calibrated.
    """
    return {name for name in ALL_CATALOG_SOURCES if name in plan_text}


In [ ]:
# NOTE: REQUIRES API KEY - ground-truth eval: 3 scenarios, up to ~27 model calls.

GT_CHECKPOINT = "gt_checkpoint.json"

def parse_sources(s: str) -> set[str]:
    return {x.strip() for x in s.split(";") if x.strip()}


def _load_checkpoint() -> dict:
    """Load previously completed scenario results from disk."""
    if Path(GT_CHECKPOINT).exists():
        with open(GT_CHECKPOINT) as f:
            data = json.load(f)
        print(f"Checkpoint found — {len(data)} scenario(s) already completed: "
              f"{list(data.keys())}")
        return data
    return {}


def _save_checkpoint(completed: dict) -> None:
    """Persist completed results to disk after each scenario."""
    with open(GT_CHECKPOINT, "w") as f:
        json.dump(completed, f)


async def evaluate_ground_truth(gt: pd.DataFrame) -> pd.DataFrame:
    completed = _load_checkpoint()
    rows = list(completed.values())  # restore any previously saved rows

    pending = [row for _, row in gt.iterrows()
               if row["Scenario"] not in completed]

    if not pending:
        print("All scenarios already completed — loaded from checkpoint.")
    else:
        print(f"{len(pending)} scenario(s) remaining.")

    for gt_row in tqdm(pending, desc="Scenarios"):
        print(f"\n  Running: {gt_row['Scenario']} ...")
        result = await run_pipeline(gt_row["QIR"], gt_row["Threat Context"],
                                    collection_temp=0.0)

        confirmed = extract_sources_from_plan(result["verified_plan"])

        expected_essential = parse_sources(gt_row["Essential Sources"])
        expected_useful    = parse_sources(gt_row["Useful Sources"])
        expected_all       = expected_essential | expected_useful

        missed_essential = expected_essential - confirmed
        essential_coverage = round(
            (len(expected_essential) - len(missed_essential))
            / len(expected_essential) * 100, 1
        ) if expected_essential else 100.0

        row = {
            "Scenario":           gt_row["Scenario"],
            "Pipeline Sources":   len(confirmed),
            "Expected Sources":   len(expected_all),
            "Overlap":            len(confirmed & expected_all),
            "Missed Essential":   len(missed_essential),
            "Essential Coverage": essential_coverage,
            "Verdict":            result["verdict"]["verdict"],
            "Iters":              result["iters"],
        }
        rows.append(row)
        completed[gt_row["Scenario"]] = row
        _save_checkpoint(completed)
        print(f"  ✓ {gt_row['Scenario']} — Essential Coverage: {essential_coverage}%")

    # All scenarios done — clean up checkpoint
    if Path(GT_CHECKPOINT).exists():
        Path(GT_CHECKPOINT).unlink()
        print("\nCheckpoint cleared.")

    return pd.DataFrame(rows)


df_gt = await evaluate_ground_truth(gt_df)
df_gt


In [ ]:
from IPython.display import display

# ── Per-scenario table ────────────────────────────────────────────────────

def _style_gt(df):
    def row_color(row):
        return (["background-color: #dcfce7"] * len(row)
                if row["Coverage"] == "100.0%"
                else ["background-color: #fee2e2"] * len(row))

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
        }.get(val, "")

    display_df = df[["Scenario", "Essential Coverage", "Missed Essential",
                      "Verdict", "Iters"]].copy()
    display_df["Essential Coverage"] = display_df["Essential Coverage"].apply(
        lambda x: f"{x:.1f}%"
    )
    display_df = display_df.rename(columns={
        "Essential Coverage": "Coverage",
        "Missed Essential":   "Missed",
    })

    styler = display_df.style.apply(row_color, axis=1)
    _map = styler.map
    return (
        _map(verdict_color, subset=["Verdict"])
        .set_caption("Ground Truth Evaluation — per-scenario results")
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.85rem"), ("padding", "8px 12px"),
                       ("border-bottom", "1px solid #f1f5f9")]},
        ])
        .hide(axis="index")
    )

display(_style_gt(df_gt))

# ── Aggregate summary ─────────────────────────────────────────────────────

THRESHOLD  = 80.0
n_run      = len(df_gt)
n_total    = len(gt_df)
is_partial = n_run < n_total

avg_coverage = df_gt["Essential Coverage"].mean()
avg_missed   = df_gt["Missed Essential"].mean()
all_passed   = (df_gt["Essential Coverage"] >= THRESHOLD).all()

print()
print(f"Scenarios run:               {n_run}/{n_total}")
print(f"Average Essential Coverage:  {avg_coverage:.1f}%")
print(f"Average Missed Essential:    {avg_missed:.1f}")
print(f"Pass rate (PASS verdict):    {(df_gt['Verdict'] == 'PASS').mean():.0%}")

status = "PASS" if all_passed else "FAIL"
note   = f" — partial run ({n_run}/{n_total} scenarios)" if is_partial else ""
print(f"\nEssential Coverage threshold ({THRESHOLD:.0f}%): {status}{note}")


## 6. Judge Calibration

Ten synthetic collection plans with known-correct verdicts test whether the Source Validation Judge correctly classifies good and bad plans. The cases cover four failure modes:

- **Hallucination detection** — real products not in the 56-source catalog (Splunk Enterprise Security, Elastic SIEM, Darktrace, Carbon Black EDR, Recorded Future)
- **Near-miss name handling** — `"Okta Logs"` vs the catalog name `"Okta System Log"`
- **Product disambiguation** — Zscaler Private Access vs Zscaler Internet Access (CASB) — different products from the same vendor
- **Relevance calibration** — Jamf Pro and Kandji MDM in an AiTM scenario — valid catalog entries but tangential to the threat

Two cases (`near_miss_names`, `relevance_tangential`) have no expected verdict — they test judge behaviour on ambiguous input, not correctness against a known answer. The effective scorable set is **8 cases**. The accuracy threshold is **80%**.

In [ ]:
# NOTE: REQUIRES API KEY - judge calibration: 10 model calls.

from judge_eval import run_all, ACCURACY_THRESHOLD

print("Running judge calibration — 10 test cases ...")
eval_results = await run_all(verbose=True)
print("Done.")


In [ ]:
df_eval = pd.DataFrame(eval_results)

def _style_eval(df):
    def row_color(row):
        if row["Scored"] == "no":
            return ["background-color: #f8fafc; color: #94a3b8"] * len(row)
        return ["background-color: #dcfce7" if row["Result"] == "✓ PASS"
                else "background-color: #fee2e2"] * len(row)

    def verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(val, "color: #64748b")

    display = df[[
        "name", "scorable", "expected_verdict", "actual_verdict",
        "expected_unverified", "actual_unverified", "passed", "summary",
    ]].copy()
    display["passed"] = display["passed"].map(
        {True: "✓ PASS", False: "✗ FAIL", None: "─ N/A"}
    )
    display["scorable"] = display["scorable"].map({True: "yes", False: "no"})
    display = display.rename(columns={
        "name":                "Test Case",
        "scorable":            "Scored",
        "expected_verdict":    "Expected",
        "actual_verdict":      "Actual",
        "expected_unverified": "Exp Unverified",
        "actual_unverified":   "Act Unverified",
        "passed":              "Result",
        "summary":             "Judge Summary",
    })
    display["Expected"]       = display["Expected"].fillna("–")
    display["Exp Unverified"] = display["Exp Unverified"].fillna("–").astype(str)

    scorable_rows  = df["scorable"]
    n_pass         = df[scorable_rows]["passed"].sum()
    n_scorable     = scorable_rows.sum()
    accuracy       = n_pass / n_scorable * 100 if n_scorable else 0.0
    status         = "PASS ✓" if accuracy >= ACCURACY_THRESHOLD else "FAIL ✗"

    styler = display.style.apply(row_color, axis=1)
    _map = styler.map
    return (
        _map(verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration — {n_pass}/{n_scorable} scorable cases correct "
            f"({accuracy:.0f}%) — Threshold {ACCURACY_THRESHOLD:.0f}% — {status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "320px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

_style_eval(df_eval)


## 7. Discussion

**On the F1 ceiling.** If grid-search F1 lands at or near 100% for every temperature, that is a sign the benchmark lacks discriminating power — not that the model is perfect. The judge validates against the same catalog the generator sees, so any catalog source the generator picks is by construction valid. To restore discriminating power: add adversarial QIRs with ambiguous product names, niche threat actors, and scenarios requiring multi-hop reasoning to identify the correct sources.

**On the judge-compliance caveat.** Grid-search F1 measures how well the generator satisfies the judge — not absolute correctness. The ground-truth evaluation is the corrective: it compares the agent's output to what a human analyst would produce, measured independently of the judge. The judge calibration test is the second corrective: it measures whether the judge itself is reliable enough to be trusted as an evaluator.

**On Monte Carlo at T=0.0.** At greedy decoding, model output is near-deterministic. Five replications at T=0.0 mostly measure infrastructure noise (API timing, kernel non-determinism), not sampling variance. Identical F1 scores across all T=0.0 simulations are expected — not a bug.

**On the two unscored calibration cases.** `near_miss_names` and `relevance_tangential` are excluded from scoring because there is no single correct answer. Whether `"Okta Logs"` should be treated as equivalent to `"Okta System Log"` is a judgment call — a strict judge rejects it, a lenient one accepts it. Recording the judge's actual behaviour on these cases is useful for characterising its calibration, even without a ground-truth verdict to score against.